# 09 · Verify the geometry of the Raptor corpus (Kaggle, CPU)

**Why?** The Raptor models of notebook 07 are trained on the author's preprocessed 44-slice corpus. At test time
the hidden DICOMs must be processed with **exactly** the same geometry, otherwise the model sees inputs it never
saw in training and the score drops silently.

The slot layout was read from the corpus mask file: **12 + 10 + 8 + 6 + 8 = 44**
(sagittal fluid, sagittal T1, coronal fluid, coronal T1, axial). Unknown: the **slice span** and the **crop (mm)**.
Here a few training studies are re-processed with candidate settings and compared pixel by pixel with the
corpus; the setting with the highest correlation is the right geometry.

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
import importlib.util, itertools
from pathlib import Path
import numpy as np, pandas as pd

from src.config import find_input
spec = importlib.util.spec_from_file_location("rk", find_input("third_party/raptor/infer_knee.py"))
rk = importlib.util.module_from_spec(spec); spec.loader.exec_module(rk)

def find(name):
    return find_input(name)
vols = np.load(find("all_vols.npy"), mmap_mode="r")
masks = np.load(find("all_masks.npy"))
ids = np.load(find("all_ids.npy"), allow_pickle=True).astype(str)
ROOT = Path(rk.find_test_root())
ser = pd.read_csv(ROOT / "train_series.csv").astype({"StudyInstanceUID": str, "SeriesInstanceUID": str})
SER = {k: v.to_dict("records") for k, v in ser.groupby("StudyInstanceUID")}
print(vols.shape, masks.shape, len(ids), ROOT)

In [ ]:
SLOTS44 = [("Sagittal", 1, 12), ("Sagittal", 0, 10), ("Coronal", 1, 8), ("Coronal", 0, 6), ("Axial", -1, 8)]
rk.SLOTS = SLOTS44; rk.MAXS = 44; rk.IMG = vols.shape[-1]
reader = rk._make_reader()
rng = np.random.default_rng(0)
rows = rng.choice(np.where(masks.sum(1) == 44)[0], 12, replace=False)  # studies with every slot filled

def corr(a, b):
    a = a.astype(np.float32).ravel(); b = b.astype(np.float32).ravel()
    return float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else np.nan

results = []
for span, crop in itertools.product([(0.06, 0.94), (0.10, 0.90), (0.15, 0.85), (0.02, 0.98)], [140.0, 120.0, 160.0]):
    rk.SPAN, rk.CROP_MM = span, crop
    cs = []
    for r in rows:
        v, m = rk.build_study(ids[r], SER, str(ROOT / "train_series"), reader)
        cs.append(np.nanmean([corr(v[i], vols[r][i]) for i in range(44) if m[i] and masks[r][i]]))
    results.append({"span": span, "crop_mm": crop, "corr_mean": np.mean(cs), "corr_min": np.min(cs)})
    print(results[-1], flush=True)
res = pd.DataFrame(results).sort_values("corr_mean", ascending=False)
res.to_csv("/kaggle/working/geometry_check.csv", index=False)
res

The top row is the right geometry (correlation ~1.0 expected; found: span 0.15–0.85, 140 mm). The final notebook (10) uses it.